In [26]:
import math
import random

import torch
import torchvision.models as models
import numpy as np
import os
from tqdm import tqdm
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
from sklearn.model_selection import train_test_split

### Data preparation

In [21]:
imgs_f = os.listdir('/home/vitya/diploma/roboarm_top_imgs2')
rwrds_f = os.listdir('/home/vitya/diploma/roboarm_top_rwrds2')

imgs = {os.path.splitext(x)[0] for x in imgs_f}
rwrds = {os.path.splitext(x)[0] for x in rwrds_f}

common_names = sorted(list(imgs & rwrds))
imgs = []
rwrds = []

for name in tqdm(common_names):
    img_name = name + '.png'
    rwrd_name = name + '.txt'

    with open(f'/home/vitya/diploma/roboarm_top_rwrds2/{rwrd_name}', 'r') as f:
        rwrd = np.array(f.read().split(',')).astype(float)[0]
    
    imgs.append(img_name)
    rwrds.append(rwrd)

data = pd.DataFrame({'images': imgs, 'labels': rwrds})

train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)
train_data.to_csv('/home/vitya/diploma/train_data.csv', index=False)
test_data.to_csv('/home/vitya/diploma/test_data.csv', index=False)

100%|██████████| 26492/26492 [00:00<00:00, 54594.81it/s]


In [22]:
class ImageRegressionDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.labels = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.labels.iloc[idx, 0])
        image = Image.open(img_path).convert('RGB')
        label = torch.tensor(float(self.labels.iloc[idx, 1]), dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [50]:
# Трансформации
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Создание датасетов
train_dataset = ImageRegressionDataset('/home/vitya/diploma/train_data.csv', '/home/vitya/diploma/roboarm_top_imgs2', transform)
val_dataset = ImageRegressionDataset('/home/vitya/diploma/test_data.csv', '/home/vitya/diploma/roboarm_top_imgs2', transform)

# Даталоадеры
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [51]:
import torch.nn as nn
from torchvision.models import mobilenet_v3_small

class MobileNetV3Regressor(nn.Module):
    def __init__(self):
        super().__init__()
        # Загрузка предобученной модели
        self.base_model = mobilenet_v3_small(weights='IMAGENET1K_V1')
        
        # Заменяем классификатор на Identity
        self.base_model.classifier = nn.Identity()
        
        # Добавляем новые слои для регрессии
        self.regressor = nn.Sequential(
            nn.Linear(576, 256),  # MobileNetV3Small output features = 576
            nn.Hardswish(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
        
    def forward(self, x):
        # Получаем features из базовой модели
        x = self.base_model(x)  # Output shape: (batch_size, 576)
        
        # Применяем регрессор
        x = self.regressor(x)  # Output shape: (batch_size, 1)
        return x

model = MobileNetV3Regressor()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [52]:
criterion = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003)

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=25):
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} train'):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, labels.unsqueeze(1))
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_loss)
        
        # Валидация
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} val'):
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                outputs = model(inputs)
                val_loss += criterion(outputs, labels.unsqueeze(1)).item() * inputs.size(0)
                
        val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(val_loss)

        torch.save(model.base_model.state_dict(), f'checkpoint_{epoch+1}.pth')
        
        print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}')

t, v = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=20)

Epoch 1/20 val: 100%|██████████| 83/83 [00:30<00:00,  2.72it/s]


Epoch 1/20, Train Loss: 0.1048, Val Loss: 0.0807


Epoch 2/20 val: 100%|██████████| 83/83 [00:30<00:00,  2.68it/s]


Epoch 2/20, Train Loss: 0.0723, Val Loss: 0.0870


Epoch 3/20 val: 100%|██████████| 83/83 [00:30<00:00,  2.71it/s]


...

Epoch 17/20, Train Loss: 0.0471, Val Loss: 0.0363


Epoch 18/20 val: 100%|██████████| 83/83 [00:32<00:00,  2.55it/s]


Epoch 18/20, Train Loss: 0.0474, Val Loss: 0.0538


Epoch 19/20 train:  95%|█████████▍| 314/332 [02:13<00:07,  2.35it/s]


KeyboardInterrupt: 

In [ ]:
torch.save(model.base_model.state_dict(), 'mobilenetv3_regressor.pth')